# What should I read next?

A reader closes a book they loved and asks your system one question: **what should I read next?** In this unit you will build the first working recommender path, search a real-schema catalog, and put both behind a scoreboard. The scoreboard matters from day one: a new idea earns its place only when it beats a seeded random baseline.

## 1. Meet the generated catalog

Before running this lesson, run the recsys catalog and interaction generators. `bookrec.generated_dir()` gives a clear message when `recsys/data/generated/` is absent.

In [ ]:
import json
from math import comb

import bookrec
import numpy as np
import pandas as pd

data_dir = bookrec.generated_dir()
catalog = bookrec.load_catalog(data_dir / "catalog.csv.gz")
len(catalog)

`load_catalog` returns a `dict[int, Book]`, keyed by stable `item_id`. A `Book` keeps `item_id` and `title` as attributes; the real generated columns `author_id`, `genres`, and `year` are strings in `Book.fields`. For analysis, make those types explicit: split the semicolon-separated genres and convert year to an integer.

In [ ]:
catalog_rows = [
    {
        "item_id": book.item_id,
        "title": book.title,
        "author_id": int(book.fields["author_id"]),
        "genres": book.fields["genres"].split(";"),
        "year": int(book.fields["year"]),
    }
    for book in catalog.values()
]
catalog_df = pd.DataFrame(catalog_rows)
catalog_df.head()

## 2. Search and filter

A catalog is useful only when we can narrow it. Boolean masks expose the mechanism: compare an attribute, test whether a genre list contains a value, and combine conditions. The generated titles are placeholders such as `Book 00000`, so text search is most meaningful over genre words and author/year digits for now; real title text arrives in later units. `search_catalog` searches a haystack containing title, genres, author-id digits, and year digits, then returns stable item-id order.

In [ ]:
recent = catalog_df[catalog_df["year"] >= 2015]
science_mask = catalog_df["genres"].apply(lambda genres: "science" in genres)
recent_science = catalog_df[(catalog_df["year"] >= 2015) & science_mask]
recent_science.head()

In [ ]:
matches = bookrec.search_catalog(catalog, text="science", year=2020)
[(book.item_id, book.title) for book in matches[:5]]

## 3. Retrieve, then rank

Every path follows one small `RetrievalPath` contract: it can `fit` or `load` its own versioned artifact, and `retrieve(query_or_user, context, k)` returns bounded `Candidate(item_id, score, provenance)` records. Equal scores break ties by item id.

```text
reader + context
       |
       v
retrieval paths --> candidates --> blend/deduplicate --> rank --> top k
```

Retrieval finds a manageable pool. Ranking orders that pool. The boundary lets later units add popularity, lexical, collaborative, and neural paths without rewriting the whole system.

In [ ]:
pool = [
    bookrec.Candidate(8, 0.70, "demo"),
    bookrec.Candidate(13, 0.95, "demo"),
    bookrec.Candidate(21, 0.80, "demo"),
]
seen = {13}
bookrec.rank(pool, n=2, exclude=seen)

## 4. A scoreboard that cannot peek into the future

The generator creates each reader's events in time order, then assigns earlier sessions to `train`, later sessions to `val`, and the final sessions to `test`. A random row split could train on a reader's future and evaluate on their past: **temporal leakage**. We tune ideas on `val`; `test` stays sealed until a later checkpoint. Readers with no unseen positive validation item and cold readers are excluded from this first scoreboard.

Each interaction row has `reader_id`, `item_id`, `session_id`, `timestamp`, `split`, and `label`. A **positive** interaction has `label == 1`. Here, a **non-cold** reader has at least one `val` row. For scoring, positive training items are already seen, so a validation re-read is removed from relevance: a re-read is not a recommendation.

In [ ]:
interactions = pd.read_csv(data_dir / "interactions.csv.gz")
split_times = interactions.groupby(["reader_id", "split"])["timestamp"].agg(["min", "max"])
ordered_times = split_times.unstack("split").dropna()
assert (ordered_times[("max", "train")] <= ordered_times[("min", "val")]).all()
assert (ordered_times[("max", "val")] <= ordered_times[("min", "test")]).all()
split_times.reset_index().sort_values(["reader_id", "min"]).head(9)

For one reader, **hit-rate@k** is 1 when at least one relevant validation book appears in the first `k`, otherwise 0. **Recall@k** is the fraction of all relevant validation books recovered in the first `k`. The scoreboard reports the mean of each per-reader number; it does not pool every event into one giant query.

In [ ]:
recommended = np.array([8, 13, 21, 34])
relevant = np.array([13, 55])
matches = np.isin(recommended[:3], relevant)
hit_at_3 = float(matches.any())
recall_at_3 = float(matches.sum() / len(relevant))
hit_at_3, recall_at_3

In [ ]:
library_hit = bookrec.hit_rate_at_k(recommended, relevant, k=3)
library_recall = bookrec.recall_at_k(recommended, relevant, k=3)
library_hit, library_recall

## 5. The seeded random floor

A baseline is the easiest system a new idea must beat. `RandomRetrievalPath` samples `k` books uniformly from the **unseen** set with NumPy's seeded generator. For exactly one relevant unseen book among `N` unseen books, symmetry gives an exact probability

```text
P(hit@k) = k / N        (for k <= N)
```

Every item occupies one of the `N` equally likely positions, and the first `k` positions are hits.

In [ ]:
random_path = bookrec.RandomRetrievalPath(catalog, seed=2026)
random_candidates = random_path.retrieve(0, {"seen": {0, 1, 2}}, k=5)
bookrec.rank(random_candidates, n=5, exclude={0, 1, 2})

In [ ]:
small_catalog = range(10)
small_seen = {0, 1}
one_relevant = 2
k = 2
trials = 2000
trial_hits = []
for seed in range(trials):
    path = bookrec.RandomRetrievalPath(small_catalog, seed=seed)
    ids = {candidate.item_id for candidate in path.retrieve(99, {"seen": small_seen}, k)}
    trial_hits.append(one_relevant in ids)
unseen_count = len(set(small_catalog) - small_seen)
analytic_floor = k / unseen_count
empirical_floor = np.mean(trial_hits)
analytic_floor, empirical_floor

### Reading the aggregate floor

Each reader has their own unseen pool of size $N_u$ and their own number of relevant unseen books $r$.
For $k$ uniform draws without replacement, the chance of **missing every** relevant book is
$\binom{N_u-r}{k}/\binom{N_u}{k}$, so a single reader's expected hit-rate@k is
$1-\binom{N_u-r}{k}/\binom{N_u}{k}$ (which collapses to $k/N_u$ when $r=1$). The scoreboard averages this over readers.

Two things to notice below. First, we demo at **k=50**, not 10: at k=10 the floor is only about 1%, which is
invisible across ~500 readers, so we raise k until the floor is clearly non-zero (it is still obviously *a floor*).
Second, the empirical number is **one random draw per reader**, so it wobbles around the analytic expectation by a
couple of points of sampling noise — seeing ~0.06 next to an expectation of ~0.04 is normal, not a bug.


In [ ]:
cold = json.loads((data_dir / "cold_partitions.json").read_text())
cold_readers = set(cold["cold_readers"])
demo_k = 50
scoreboard = bookrec.run_validation_scoreboard(
    bookrec.RandomRetrievalPath(catalog, seed=2026),
    data_dir / "interactions.csv.gz",
    catalog_ids=catalog,
    k=demo_k,
    cold_readers=cold_readers,
)
analytic_hits = []
val_positive = interactions[(interactions["split"] == "val") & (interactions["label"] == 1)]
for reader_id in sorted(set(val_positive["reader_id"]) - cold_readers):
    reader_rows = interactions[interactions["reader_id"] == reader_id]
    seen = set(reader_rows[(reader_rows["split"] == "train") & (reader_rows["label"] == 1)]["item_id"])
    relevant = set(reader_rows[(reader_rows["split"] == "val") & (reader_rows["label"] == 1)]["item_id"]) - seen
    if relevant:
        unseen_count = len(catalog) - len(seen)
        miss_probability = comb(unseen_count - len(relevant), demo_k) / comb(unseen_count, demo_k)
        analytic_hits.append(1 - miss_probability)
analytic_expected_hit_rate = float(np.mean(analytic_hits))
{
    "empirical_hit_rate_at_50": scoreboard.hit_rate_at_k,
    "analytic_expected_hit_rate_at_50": analytic_expected_hit_rate,
    "empirical_recall_at_50": scoreboard.recall_at_k,
}

The system now has a catalog, search, a stable retrieve-then-rank contract, and a leakage-safe validation scoreboard. In Unit 2, a popularity path will face its first honest test: **can it beat random?**